In [1]:
import os
import glob
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# ----------------------------
# Custom Dataset (Part 1)
# ----------------------------
class CustomDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        # Search for JPG images in the specified directory.
        self.image_paths = glob.glob(os.path.join(image_dir, '*.jpg'))
        print(f"Found {len(self.image_paths)} images in {image_dir}")
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Load an image and convert it to RGB
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        # Apply transformations if provided (e.g., resize, to tensor)
        if self.transform:
            image = self.transform(image)
        # In our demo, the same image serves as both low-res input and high-res target.
        return {'low_res': image, 'high_res': image}

# ----------------------------
# Image Transformations
# ----------------------------
# Resize images to 299x299 and convert them to tensors.
transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
])

# ----------------------------
# DataLoader Setup
# ----------------------------
# Set the directory where your resized images are stored.
resized_dir = '/kaggle/working/resized_images'
# Create an instance of the custom dataset.
train_dataset = CustomDataset(resized_dir, transform=transform)
# Create a DataLoader to iterate through the dataset.
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

# ----------------------------
# Debug: Verify DataLoader Output
# ----------------------------
# Iterate over one batch and print the keys and shapes.
for batch in train_loader:
    print("Batch keys:", batch.keys())
    print("Low-res image batch shape:", batch['low_res'].shape)  # Expected: [4, 3, 299, 299]
    break

Found 1887 images in /kaggle/working/resized_images
Batch keys: dict_keys(['low_res', 'high_res'])
Low-res image batch shape: torch.Size([4, 3, 299, 299])


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ----------------------------
# SRCNN Model Definition
# ----------------------------
class SRCNN(nn.Module):
    def __init__(self):
        super(SRCNN, self).__init__()
        # First layer: 9x9 kernel to capture context
        self.conv1 = nn.Conv2d(3, 64, kernel_size=9, padding=4)
        # Second layer: 5x5 kernel to refine features
        self.conv2 = nn.Conv2d(64, 32, kernel_size=5, padding=2)
        # Third layer: 5x5 kernel to reconstruct the output (RGB)
        self.conv3 = nn.Conv2d(32, 3, kernel_size=5, padding=2)
    
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.conv3(x)
        return x

# ----------------------------
# U-Net Block Definition
# ----------------------------
class UNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UNetBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        return x

# ----------------------------
# Center Crop Function for Skip Connections
# ----------------------------
def center_crop(tensor, target_size):
    """
    Center crop a tensor to target_size (height, width).
    Assumes tensor shape is (B, C, H, W).
    """
    _, _, h, w = tensor.size()
    target_h, target_w = target_size
    delta_h = h - target_h
    delta_w = w - target_w
    top = delta_h // 2
    left = delta_w // 2
    return tensor[:, :, top:top+target_h, left:left+target_w]

# ----------------------------
# Dual-Input U-Net Definition
# ----------------------------
class DualInputUNet(nn.Module):
    def __init__(self, input_channels=6, output_channels=3):
        super(DualInputUNet, self).__init__()
        # Encoder: process the concatenated inputs (original + SRCNN output)
        self.enc1 = UNetBlock(input_channels, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = UNetBlock(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        
        # Bottleneck
        self.bottleneck = UNetBlock(128, 256)
        
        # Decoder: upsampling with skip connections
        self.up1 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec1 = UNetBlock(256, 128)  # Concatenates up1 (128 channels) with cropped enc2 (128 channels)
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = UNetBlock(128, 64)   # Concatenates up2 (64 channels) with cropped enc1 (64 channels)
        
        # Final output layer to get a 3-channel enhanced image
        self.final_conv = nn.Conv2d(64, output_channels, kernel_size=1)
    
    def forward(self, orig, super_res):
        # Concatenate original image and SRCNN output along channel dimension: [B, 6, H, W]
        x = torch.cat([orig, super_res], dim=1)
        
        # Encoder path
        enc1 = self.enc1(x)                      # Output: [B, 64, H, W]
        enc2 = self.enc2(self.pool1(enc1))         # Output: [B, 128, H/2, W/2]
        
        # Bottleneck
        bottleneck = self.bottleneck(self.pool2(enc2))  # Output: [B, 256, H/4, W/4]
        
        # Decoder path
        up1 = self.up1(bottleneck)              # Upsample: [B, 128, H/2, W/2]
        enc2_cropped = center_crop(enc2, up1.shape[2:])  # Crop to match dimensions
        dec1 = self.dec1(torch.cat([up1, enc2_cropped], dim=1))  # [B, 128, H/2, W/2]
        
        up2 = self.up2(dec1)                    # Upsample: [B, 64, H, W]
        enc1_cropped = center_crop(enc1, up2.shape[2:])  # Crop to match dimensions
        dec2 = self.dec2(torch.cat([up2, enc1_cropped], dim=1))  # [B, 64, H, W]
        
        out = self.final_conv(dec2)             # Final output: [B, 3, H, W]
        return out

# ----------------------------
# Testing the Models with Dummy Data (Debugging)
# ----------------------------
if __name__ == '__main__':
    # Create a dummy input tensor: [1, 3, 299, 299]
    dummy_input = torch.randn(1, 3, 299, 299)
    
    # Test SRCNN forward pass
    srcnn_model = SRCNN()
    srcnn_output = srcnn_model(dummy_input)
    print("SRCNN output shape:", srcnn_output.shape)  # Expected: [1, 3, 299, 299]
    
    # Test Dual-Input U-Net forward pass using dummy input and SRCNN output
    unet_model = DualInputUNet()
    enhanced_output = unet_model(dummy_input, srcnn_output)
    print("Dual-Input U-Net enhanced output shape:", enhanced_output.shape)

SRCNN output shape: torch.Size([1, 3, 299, 299])
Dual-Input U-Net enhanced output shape: torch.Size([1, 3, 296, 296])


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import DataLoader
import glob
import os
from PIL import Image

# ----------------------------
# Custom Dataset (from Part 1)
# ----------------------------
class CustomDataset(object):
    def __init__(self, image_dir, transform=None):
        self.image_paths = glob.glob(os.path.join(image_dir, '*.jpg'))
        print(f"Found {len(self.image_paths)} images in {image_dir}")
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return {'low_res': image, 'high_res': image}

# ----------------------------
# Image Transformations
# ----------------------------
transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
])

# Set your directory (update if necessary)
resized_dir = '/kaggle/working/resized_images'
train_dataset = CustomDataset(resized_dir, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

# ----------------------------
# Model Definitions (from Part 2)
# ----------------------------
# SRCNN Definition
class SRCNN(nn.Module):
    def __init__(self):
        super(SRCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=9, padding=4)
        self.conv2 = nn.Conv2d(64, 32, kernel_size=5, padding=2)
        self.conv3 = nn.Conv2d(32, 3, kernel_size=5, padding=2)
    
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.conv3(x)
        return x

# U-Net Block Definition
class UNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UNetBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        return x

# Center Crop Function (our custom function)
def center_crop(tensor, target_size):
    """
    Center crop a tensor to target_size (height, width).
    Assumes tensor shape is (B, C, H, W).
    """
    _, _, h, w = tensor.size()
    target_h, target_w = target_size
    delta_h = h - target_h
    delta_w = w - target_w
    top = delta_h // 2
    left = delta_w // 2
    return tensor[:, :, top:top+target_h, left:left+target_w]

# Dual-Input U-Net Definition
class DualInputUNet(nn.Module):
    def __init__(self, input_channels=6, output_channels=3):
        super(DualInputUNet, self).__init__()
        self.enc1 = UNetBlock(input_channels, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = UNetBlock(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        self.bottleneck = UNetBlock(128, 256)
        self.up1 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec1 = UNetBlock(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = UNetBlock(128, 64)
        self.final_conv = nn.Conv2d(64, output_channels, kernel_size=1)
    
    def forward(self, orig, super_res):
        x = torch.cat([orig, super_res], dim=1)
        enc1 = self.enc1(x)
        enc2 = self.enc2(self.pool1(enc1))
        bottleneck = self.bottleneck(self.pool2(enc2))
        up1 = self.up1(bottleneck)
        enc2_cropped = center_crop(enc2, up1.shape[2:])
        dec1 = self.dec1(torch.cat([up1, enc2_cropped], dim=1))
        up2 = self.up2(dec1)
        enc1_cropped = center_crop(enc1, up2.shape[2:])
        dec2 = self.dec2(torch.cat([up2, enc1_cropped], dim=1))
        out = self.final_conv(dec2)
        return out

# ----------------------------
# Custom Loss Functions
# ----------------------------
def edge_loss(output, target):
    """
    Compute an edge-aware loss using a Sobel filter.
    Measures the L1 difference between gradients of the output and target.
    """
    sobel_kernel = torch.tensor([[[[-1, 0, 1],
                                   [-2, 0, 2],
                                   [-1, 0, 1]]]], dtype=torch.float32, device=output.device)
    channels = output.size(1)
    sobel_kernel = sobel_kernel.repeat(channels, 1, 1, 1)
    
    grad_output = F.conv2d(output, sobel_kernel, padding=1, groups=channels)
    grad_target = F.conv2d(target, sobel_kernel, padding=1, groups=channels)
    
    return F.l1_loss(grad_output, grad_target)

def combined_loss(output, target, alpha=1.0, beta=1.0, gamma=1.0):
    loss_l1 = F.l1_loss(output, target)
    loss_mse = F.mse_loss(output, target)
    loss_edge = edge_loss(output, target)
    return alpha * loss_l1 + beta * loss_mse + gamma * loss_edge

# ----------------------------
# Setup Models and Optimizer
# ----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
srcnn_model = SRCNN().to(device)
unet_model = DualInputUNet().to(device)

optimizer = optim.Adam(list(srcnn_model.parameters()) + list(unet_model.parameters()), lr=1e-4)

# ----------------------------
# Training Loop (Full Image Version)
# ----------------------------
num_epochs = 3  # For debugging purposes
for epoch in range(num_epochs):
    srcnn_model.train()
    unet_model.train()
    running_loss = 0.0
    
    for batch in train_loader:
        low_res = batch['low_res'].to(device)   # Shape: [B, 3, 299, 299]
        high_res = batch['high_res'].to(device)   # Shape: [B, 3, 299, 299]
        
        optimizer.zero_grad()
        
        # Generate super-resolved image with SRCNN
        super_res = srcnn_model(low_res)
        
        # Generate enhanced image with Dual-Input U-Net
        enhanced = unet_model(low_res, super_res)
        
        # U-Net output might be slightly smaller due to cropping; adjust high_res accordingly.
        _, _, H, W = enhanced.size()
        high_res_cropped = center_crop(high_res, [H, W])
        
        # Compute the combined loss
        loss = combined_loss(enhanced, high_res_cropped)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

# Optionally, save the model weights
torch.save(srcnn_model.state_dict(), "srcnn_full_image.pth")
torch.save(unet_model.state_dict(), "unet_full_image.pth")

Found 1887 images in /kaggle/working/resized_images
Epoch 1/3, Loss: 0.1071
Epoch 2/3, Loss: 0.0238
Epoch 3/3, Loss: 0.0184


In [ ]:
import os
import glob
import torch
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
from skimage import filters, metrics
import torch.nn as nn
import torch.nn.functional as F

# ----------------------------
# Model Definitions
# ----------------------------
class SRCNN(nn.Module):
    def __init__(self):
        super(SRCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=9, padding=4)
        self.conv2 = nn.Conv2d(64, 32, kernel_size=5, padding=2)
        self.conv3 = nn.Conv2d(32, 3, kernel_size=5, padding=2)
    
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.conv3(x)
        return x

class UNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UNetBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        return x

def center_crop(tensor, target_size):
    # Assumes tensor shape: (B, C, H, W)
    _, _, h, w = tensor.size()
    target_h, target_w = target_size
    top = (h - target_h) // 2
    left = (w - target_w) // 2
    return tensor[:, :, top:top+target_h, left:left+target_w]

class DualInputUNet(nn.Module):
    def __init__(self, input_channels=6, output_channels=3):
        super(DualInputUNet, self).__init__()
        self.enc1 = UNetBlock(input_channels, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = UNetBlock(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        self.bottleneck = UNetBlock(128, 256)
        self.up1 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec1 = UNetBlock(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = UNetBlock(128, 64)
        self.final_conv = nn.Conv2d(64, output_channels, kernel_size=1)
    
    def forward(self, orig, super_res):
        x = torch.cat([orig, super_res], dim=1)
        enc1 = self.enc1(x)
        enc2 = self.enc2(self.pool1(enc1))
        bottleneck = self.bottleneck(self.pool2(enc2))
        up1 = self.up1(bottleneck)
        enc2_cropped = center_crop(enc2, up1.shape[2:])
        dec1 = self.dec1(torch.cat([up1, enc2_cropped], dim=1))
        up2 = self.up2(dec1)
        enc1_cropped = center_crop(enc1, up2.shape[2:])
        dec2 = self.dec2(torch.cat([up2, enc1_cropped], dim=1))
        out = self.final_conv(dec2)
        return out

# ----------------------------
# Load Trained Models
# ----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
srcnn_model = SRCNN().to(device)
unet_model = DualInputUNet().to(device)

# Update these paths if necessary.
srcnn_model.load_state_dict(torch.load("srcnn_full_image.pth", map_location=device))
unet_model.load_state_dict(torch.load("unet_full_image.pth", map_location=device))
srcnn_model.eval()
unet_model.eval()

# ----------------------------
# Image Preprocessing
# ----------------------------
transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
])
to_pil = transforms.ToPILImage()

# ----------------------------
# Process and Evaluate Multiple Images
# ----------------------------
# Get a list of image paths from the resized folder (limit to at least 5 images)
resized_dir = "/kaggle/working/resized_images"
all_image_paths = glob.glob(os.path.join(resized_dir, '*.jpg'))
num_eval = 5
test_image_paths = all_image_paths[:num_eval]  # Use first 5 images

# Lists to store computed metrics and images for visualization
original_images = []
enhanced_images = []
original_edge_maps = []
enhanced_edge_maps = []
psnr_list = []
ssim_list = []

for img_path in test_image_paths:
    # Load and preprocess image
    image = Image.open(img_path).convert("RGB")
    image = image.resize((299, 299))
    image_tensor = transform(image).unsqueeze(0).to(device)  # [1, 3, 299, 299]
    
    with torch.no_grad():
        # Process through models
        sr_tensor = srcnn_model(image_tensor)
        enhanced_tensor = unet_model(image_tensor, sr_tensor)
    
    # Because U-Net may crop the output, adjust original image to match
    _, _, H, W = enhanced_tensor.size()
    image_cropped = torch.nn.functional.interpolate(image_tensor, size=(H, W), mode="bilinear", align_corners=False)
    
    # Convert tensors to numpy arrays for visualization
    def tensor_to_np(tensor):
        np_img = tensor.squeeze(0).cpu().clamp(0, 1).numpy()  # [C, H, W]
        np_img = np.transpose(np_img, (1, 2, 0))  # [H, W, C]
        return np_img
    
    original_np = tensor_to_np(image_cropped)
    enhanced_np = tensor_to_np(enhanced_tensor)
    
    # Compute edge maps (Sobel filter on grayscale by averaging channels)
    original_edges = filters.sobel(np.mean(original_np, axis=2))
    enhanced_edges = filters.sobel(np.mean(enhanced_np, axis=2))
    
    # Compute metrics with explicit win_size and using channel_axis for multichannel images.
    psnr_val = metrics.peak_signal_noise_ratio(original_np, enhanced_np, data_range=1)
    ssim_val = metrics.structural_similarity(original_np, enhanced_np, win_size=7, channel_axis=-1, data_range=1)
    
    # Append for later visualization
    original_images.append(original_np)
    enhanced_images.append(enhanced_np)
    original_edge_maps.append(original_edges)
    enhanced_edge_maps.append(enhanced_edges)
    psnr_list.append(psnr_val)
    ssim_list.append(ssim_val)

# ----------------------------
# Visualization: Display Results for Each Image
# ----------------------------
num_images = len(original_images)
fig, axes = plt.subplots(num_images, 4, figsize=(20, 5*num_images))

for i in range(num_images):
    # If only one image, axes will not be 2D; ensure it's 2D.
    if num_images == 1:
        row_axes = axes
    else:
        row_axes = axes[i]
    
    row_axes[0].imshow(original_images[i])
    row_axes[0].set_title("Original")
    row_axes[0].axis("off")
    
    row_axes[1].imshow(enhanced_images[i])
    row_axes[1].set_title("Enhanced")
    row_axes[1].axis("off")
    
    row_axes[2].imshow(original_edge_maps[i], cmap="gray")
    row_axes[2].set_title("Original Edges")
    row_axes[2].axis("off")
    
    row_axes[3].imshow(enhanced_edge_maps[i], cmap="gray")
    row_axes[3].set_title("Enhanced Edges")
    row_axes[3].axis("off")
    
    # Add a super title for each row with metrics
    row_axes[0].set_ylabel(f"PSNR: {psnr_list[i]:.2f} dB\nSSIM: {ssim_list[i]:.2f}", fontsize=14)

plt.tight_layout()
plt.show()